In [0]:
import pandas as pd
import numpy as np
import time
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# 1. Generar datos sintéticos grandes (20 millones de filas)
print("Generando datos...")
n_rows = 40000000
pdf = pd.DataFrame({
    'categoria': np.random.choice(['A', 'B', 'C', 'D', 'E'], n_rows),
    'valor1': np.random.rand(n_rows),
    'valor2': np.random.rand(n_rows) * 100
})

# Convertir a Spark DataFrame
sdf = spark.createDataFrame(pdf)
# Forzar a que Spark lo ponga en memoria/caché para una comparación justa
sdf.cache().count() 

print("--- Iniciando Comparación ---")

# 2. Prueba con Pandas
start_time = time.time()
pdf_result = pdf.groupby('categoria').agg({'valor1': 'mean', 'valor2': 'sum'})
pandas_time = time.time() - start_time
print(f"Tiempo de ejecución en Pandas (Single Node): {pandas_time:.2f} segundos")

# 3. Prueba con Spark
start_time = time.time()
sdf_result = sdf.groupBy('categoria').agg(
    F.mean('valor1').alias('mean_valor1'), 
    F.sum('valor2').alias('sum_valor2')
)
sdf_result.collect() # Acción para forzar el cómputo
spark_time = time.time() - start_time
print(f"Tiempo de ejecución en Spark (Distribuido/Serverless): {spark_time:.2f} segundos")

print(f"\n¡Spark es aprox {pandas_time / spark_time:.1f}x más rápido en esta operación básica!")